# **Import Library Example**

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.functions import avg, col
from pyspark.sql.functions import month, year, sum as sum_
from pyspark.sql.functions import sum as sum_
from pyspark.sql.functions import col, sum as sum_, count
import matplotlib.pyplot as plt
from pyspark.sql.functions import sum
import pandas as pd
from pyspark.sql.functions import when
from pyspark.sql.functions import lit
import numpy as np
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier, LogisticRegression
from pyspark.ml.regression import LinearRegression
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, RegressionEvaluator
from pyspark.ml.evaluation import ClusteringEvaluator
import seaborn as sns
import mlflow

# Data Loading and Import

In this step, the dataset is loaded into the Databricks environment using Apache Spark. The dataset is read from the specified file path in CSV format, with headers enabled and schema automatically inferred.

This process converts the raw dataset into a Spark DataFrame, which allows efficient distributed data processing and analysis.

In [0]:
df = spark.read.csv("/Workspace/Users/heshanisasindi@gmail.com/Dataset/online_retail_II.csv", header=True, inferSchema=True)

# Dataset Overview

After loading the dataset, an initial exploration is performed to understand its structure, size, and attributes. This step is important to gain a clear understanding of the data before proceeding with preprocessing and analysis.

In [0]:
print("Row count:", df.count())
print("Column count:", len(df.columns))
print("Columns:", df.columns)

Row count: 1067371
Column count: 8
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [0]:
display(df.limit(10))

Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01T07:45:00.000Z,6.95,13085.0,United Kingdom
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom
489434,22041,"""RECORD FRAME 7"""" SINGLE SIZE """,48,2009-12-01T07:45:00.000Z,2.1,13085.0,United Kingdom
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom
489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01T07:45:00.000Z,1.65,13085.0,United Kingdom
489434,21871,SAVE THE PLANET MUG,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom
489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01T07:45:00.000Z,5.95,13085.0,United Kingdom
489435,22350,CAT BOWL,12,2009-12-01T07:46:00.000Z,2.55,13085.0,United Kingdom
489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01T07:46:00.000Z,3.75,13085.0,United Kingdom


# Predictive Analytics using Spark ML 

In [0]:
# Prepare features for classification (using only numeric features to avoid model caching)
assembler = VectorAssembler(inputCols=["Quantity", "UnitPrice"], outputCol="features")
df_ml = assembler.transform(df_clean)

In [0]:
# Split data
train, test = df_ml.randomSplit([0.7, 0.3], seed=42)

## Decision Tree Classifier

In [0]:
# Decision Tree Classifier
# Create a binary label without fitting models (to avoid cache overflow)
train_labeled = train.withColumn("label", when(train["Sales"] > 50, 1).otherwise(0))
test_labeled = test.withColumn("label", when(test["Sales"] > 50, 1).otherwise(0))

dt = DecisionTreeClassifier(labelCol="label", featuresCol="features")
dt_model = dt.fit(train_labeled)
dt_preds = dt_model.transform(test_labeled)

##  Random Forest Classifier 

In [0]:
# Random Forest Classifier (using same label as Decision Tree)
rf = RandomForestClassifier(labelCol="label", featuresCol="features")
rf_model = rf.fit(train_labeled)
rf_preds = rf_model.transform(test_labeled)

##  Logistic Regression

In [0]:
# Logistic Regression
lr = LogisticRegression(labelCol="label", featuresCol="features", maxIter=10)
lr_model = lr.fit(train_labeled)
lr_preds = lr_model.transform(test_labeled)

## Classification Evaluation

In [0]:

# Classification Evaluation
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")
evaluator_prec = MulticlassClassificationEvaluator(labelCol="label", metricName="weightedPrecision")
evaluator_rec = MulticlassClassificationEvaluator(labelCol="label", metricName="weightedRecall")

results_classification = [
    ["DecisionTree", evaluator_acc.evaluate(dt_preds), evaluator_prec.evaluate(dt_preds), evaluator_rec.evaluate(dt_preds)],
    ["RandomForest", evaluator_acc.evaluate(rf_preds), evaluator_prec.evaluate(rf_preds), evaluator_rec.evaluate(rf_preds)],
    ["LogisticRegression", evaluator_acc.evaluate(lr_preds), evaluator_prec.evaluate(lr_preds), evaluator_rec.evaluate(lr_preds)]
]

## Regression- Predict 'Sales'

In [0]:
# Regression - Predict 'Sales'
assembler_reg = VectorAssembler(inputCols=["Quantity", "UnitPrice"], outputCol="features")
df_reg = assembler_reg.transform(df_clean)
train_reg, test_reg = df_reg.randomSplit([0.7, 0.3], seed=42)
lr_reg = LinearRegression(featuresCol="features", labelCol="Sales")
lr_reg_model = lr_reg.fit(train_reg)
lr_reg_preds = lr_reg_model.transform(test_reg)

##  Regression Evaluation

In [0]:

# Regression Evaluation
reg_evaluator_mae = RegressionEvaluator(labelCol="Sales", predictionCol="prediction", metricName="mae")
reg_evaluator_rmse = RegressionEvaluator(labelCol="Sales", predictionCol="prediction", metricName="rmse")
results_regression = [
    ["LinearRegression", reg_evaluator_mae.evaluate(lr_reg_preds), reg_evaluator_rmse.evaluate(lr_reg_preds)]
]